# JSON Schema形式の使用

例1：

In [4]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# .envファイルから環境変数を読み込む
load_dotenv(override=True)

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_API_BASE = os.getenv("OPENROUTER_API_BASE")

model = init_chat_model(
    model="openai/gpt-4o-mini",
    model_provider="openai",
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_API_BASE,
)

In [5]:
json_schema = {
    "title": "Movie",
    "description": "A movie with details",
    "type": "object",
    "properties": {
        "title": {
            "type": "string",
            "description": "The title of the movie"
        },
        "year": {
            "type": "integer",
            "description": "The year the movie was released"
        },
        "director": {
            "type": "string",
            "description": "The director of the movie"
        },
        "rating": {
            "type": "number",
            "description": "The movie's rating out of 10"
        }
    },
    "required": ["title", "year", "director", "rating"]
}


structured_model = model.with_structured_output(json_schema,method="json_schema")
response = structured_model.invoke("インセプションの情報を教えてください")
print(response)
print(type(response))

{'title': 'インセプション', 'year': 2010, 'director': 'クリストファー・ノーラン', 'rating': 8.8}
<class 'dict'>


例2：

In [6]:
"""
JSON Schema を使ってネスト構造を定義する
"""
# 1. ネストされた JSON Schema を定義
project_schema = {
    "title": "MovieInfo",
    "description": "映画タイトル、公開年、監督、キャスト、評価を含む映画オブジェクト",
    "type": "object",
    "properties": {
        "title": {"type": "string", "description": "映画タイトル"},
        "year": {"type": "integer", "description": "公開年"},
        "director": {"type": "string", "description": "監督"},
        "cast": {  # ネストされた配列を定義
            "type": "array",
            "description": "俳優リスト",
            "items": {
                "type": "object",
                "properties": {
                    "name": {"type": "string", "description": "俳優名"},
                    "role": {"type": "string", "description": "演じる役柄"}
                },
                "required": ["name", "role"]
            }
        },
        "rating": {"type": "number", "description": "評価（10点満点）"}
    },
    "required": ["title", "year", "director", "cast", "rating"]
}

# JSON Schema をモデルにバインド
structured_model = model.with_structured_output(project_schema,method="json_schema")

# モデルを呼び出し
response = structured_model.invoke("映画『インターステラー』についての情報を生成してください。監督、キャスト、評価を含めてください")
print(response)

{'title': 'インターステラー', 'year': 2014, 'director': 'クリストファー・ノーラン', 'cast': [{'name': 'マシュー・マコノヒー', 'role': 'クーパー'}, {'name': 'アン・ハサウェイ', 'role': 'ブランド'}, {'name': 'ジェシカ・チャステイン', 'role': 'マーサ・クーパー'}, {'name': 'マイケル・ケイン', 'role': 'ドクター・ブランド'}, {'name': 'ティム・ロビンス', 'role': 'クエンティン'}], 'rating': 8.6}
